# AliasingAtlas Notebook

> Interactive playground for sampling, aliasing, reconstruction, and audio export.

This notebook launches the same UI shipped by the package and is aligned with the latest features. **Compatible with local Jupyter and Google Colab** (auto-installs via pip if needed).

### What You Can Explore
- Nyquist and aliasing behavior with live spectra and fold indicators.
- Reconstruction comparison (**FFT** vs **FOH**).
- Anti-alias filtering (**None / Ideal / Butter**).
- Quantization effects and error plot.
- Preset scenarios (Safe Nyquist, Near Nyquist, Aliasing, AM Sidebands).
- Guided learning hints (**Learning: On/Off**).
- Richer waveforms including **Chirp**.
- Export toolkit from the Audio panel (**Export CFG**, **Export WAV**).

### Getting Started
- **Local:** Ensure you're in the project root directory (where `src/` exists)
- **Colab:** Just run the cells—automatic pip installation is handled

### Tip
For the best experience, run all cells in order and keep the interactive backend enabled.

In [ ]:
# Notebook setup (local + Colab-friendly)
from pathlib import Path
import os
import sys
import subprocess

is_colab = False
try:
    import google.colab  # type: ignore
    is_colab = True
    # Install visualization dependencies (skip sounddevice - PortAudio not available)
    !pip install -q ipympl
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

try:
    get_ipython().run_line_magic("matplotlib", "widget")
except Exception:
    get_ipython().run_line_magic("matplotlib", "inline")


def add_src_to_path() -> None:
    """Add src directory to path if running locally, handle Colab specially."""
    if is_colab:
        # In Colab, try to clone from GitHub if src not available
        repo_root = Path.cwd()
        candidate = repo_root / "src"
        
        if (candidate / "aliasing_atlas").exists():
            sys.path.insert(0, str(candidate.resolve()))
            return
        
        # Clone from GitHub if not found locally
        print("Installing AliasingAtlas from GitHub...")
        try:
            subprocess.run(
                ["git", "clone", "--depth", "1", 
                 "https://github.com/Boussetta/NyquistNavigator.git",
                 "nyquist_nav"],
                check=True,
                capture_output=True
            )
            sys.path.insert(0, "/content/nyquist_nav/src")
            print("✓ Successfully cloned and added to path")
            return
        except Exception as e:
            print(f"⚠ Could not clone from GitHub: {e}")
            print("Attempting to use local path detection...")
    
    repo_root = Path.cwd()
    
    # Try direct src path
    candidate = repo_root / "src"
    if (candidate / "aliasing_atlas").exists():
        sys.path.insert(0, str(candidate.resolve()))
        return

    # Try walking up directory tree
    for root, dirs, _ in os.walk(repo_root):
        if "aliasing_atlas" in dirs and Path(root).name == "src":
            sys.path.insert(0, str(Path(root).resolve()))
            return
    
    # Try parent directories
    current = repo_root
    for _ in range(5):
        candidate = current / "src" / "aliasing_atlas"
        if candidate.exists():
            sys.path.insert(0, str((current / "src").resolve()))
            return
        current = current.parent

    raise FileNotFoundError(
        "Could not find aliasing_atlas module.\n"
        "• Local: Run from project root where src/ exists\n"
        "• Colab: Ensure git is available or manually clone from:\n"
        "  https://github.com/Boussetta/NyquistNavigator.git"
    )


add_src_to_path()

from aliasing_atlas.app import AliasingToolbox

In [ ]:
# Launch the interactive toolbox
toolbox = AliasingToolbox()
toolbox.show()

# Exports from the UI are written under ./exports/